# 07장. 테스트와 모델 카드

| 오늘의 질문 | 예상 시간 |
|---|---:|
| 추천기를 어떻게 시험하고 한계를 설명할까? | 5회차 · 약 180분 |


## 이 장에서 배울 내용

- 테스트의 예상·실행·판정 구조를 설명할 수 있다.
- 입력을 바꾼 네 가지 사례로 추천 규칙을 확인할 수 있다.
- 모델 카드에 목적·데이터·방법·한계·금지 사용을 기록할 수 있다.


## 생각 열기

추천 버튼이 한 번 작동한 것만으로는 충분하지 않습니다. 취향이 비어 있거나 범위를 벗어난 값이 들어왔을 때도 정해 둔 방식으로 움직이는지 확인합니다. 모델 카드에는 시험 결과와 함께 이 추천기가 하지 못하는 일도 적습니다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **테스트 사례** | 특정 입력과 예상 결과를 짝지은 시험 |
| **회귀** | 고친 기능이 이후 변경으로 다시 망가지는 현상 |
| **모델 카드** | AI의 목적, 데이터, 방법, 한계를 공개하는 설명서 |
| **책임 있는 AI** | 사람에게 미칠 영향과 한계를 고려하는 개발 태도 |


## 개념 익히기


과학 실험은 예상하고, 실행하고, 결과를 비교합니다. 소프트웨어 테스트도 같습니다. ‘오이를 피하면 오이 메뉴 점수가 내려간다’처럼 구체적인 예상이 있어야 판정할 수 있습니다.

모델 카드는 제품 설명서와 비슷합니다. 잘하는 것만 쓰지 않고 데이터가 적다는 점, 실제 만족도를 모른다는 점, 의료 판단에 쓰면 안 된다는 점도 함께 공개합니다.


## 활동 전 생각


‘좋아함을 파스타로 입력한다’라는 시험에 대해 예상 결과를 한 문장으로 적으세요. ‘잘 된다’보다 ‘파스타가 포함된 메뉴의 점수 또는 이유가 상승한다’가 더 좋은 예상입니다.


## 예상하기

- 네 가지 시험 사례가 모두 통과한다.
- 모델 카드의 다섯 필수 항목이 채워진다.


## 활동 1. 네 가지 추천 시험


### 코드 살펴보기


1. `test_records`에는 시험 이름과 통과 여부를 한 쌍으로 저장합니다.<br>
2. `plain`, `liked`, `fake_allergy`는 서로 다른 입력 조건의 추천 결과입니다.<br>
3. `assert all(...)`은 네 시험 중 하나라도 실패하면 실행을 멈춥니다.


In [ ]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. 프로젝트 최상위 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

from neis_meal_ai.recommender import PreferenceProfile, recommend_menus, validate_profile

test_records = []

plain = recommend_menus(meal_df, PreferenceProfile((), (), (), 2, ()), top_n=3)
test_records.append(("빈 취향도 실행", len(plain) == 3))

liked = recommend_menus(
    meal_df, PreferenceProfile(("파스타",), (), ("면",), 2, ()), top_n=3
)
test_records.append(("파스타 좋아함 이유 표시", "파스타" in liked.iloc[0]["reason"]))

fake_allergy = recommend_menus(
    meal_df, PreferenceProfile((), (), (), 2, (1,)), top_n=5
)
test_records.append((
    "가상 번호 1 메뉴 제외",
    all(1 not in codes for codes in fake_allergy["allergy_codes"]),
))

try:
    validate_profile(PreferenceProfile((), (), (), 7, ()))
    invalid_spice_rejected = False
except ValueError:
    invalid_spice_rejected = True
test_records.append(("범위 밖 매운맛 차단", invalid_spice_rejected))

for name, passed in test_records:
    print("PASS" if passed else "FAIL", "-", name)
assert all(passed for _, passed in test_records)


### 결과 해석하기

각 시험은 바뀐 입력이 어떤 결과를 만들어야 하는지 확인합니다. 실패가 나오면 발표 전에 원인을 찾아야 합니다.


## 활동 2. 모델 카드 만들기


### 코드 살펴보기


1. `model_card`는 목적, 데이터, 방법, 한계, 금지 사용, 안전 문구를 한 사전에 모읍니다.<br>
2. `items()`로 각 항목을 빠짐없이 출력합니다.<br>
3. `all(bool(value) ...)`는 빈 항목이 있는지 확인합니다.


In [ ]:
model_card = {
    "목적": "익명 가상 취향과 NEIS 메뉴 표현을 비교한 상대 추천",
    "데이터": "남악고 NEIS 공개 급식 예비 데이터 5행",
    "방법": "문자 n-gram TF-IDF, 코사인 유사도, 공개 가감점, 작은 K-Means",
    "한계": "실제 만족도, 숨은 재료, 개인 건강 적합도를 알 수 없음",
    "금지 사용": "의료 판단, 실제 알레르기 안전 결정, 학생 평가",
    "안전 문구": (
        "추천 결과는 취향 비교용입니다. 실제 식단과 알레르기 정보는 "
        "학교 급식표와 영양사 안내를 다시 확인하세요."
    ),
}
for key, value in model_card.items():
    print(f"- {key}: {value}")

chapter_result = {
    "chapter": "07",
    "tests_passed": sum(passed for _, passed in test_records),
    "model_card_complete": all(bool(value) for value in model_card.values()),
}


### 결과 해석하기

모델 카드는 AI를 과장하지 않도록 목적과 금지 사용을 동시에 보여 줍니다. 발표 화면에도 안전 문구를 남깁니다.


## 탐구 활동

아래 팀 한계 문장을 메뉴 데이터의 부족한 점이 드러나도록 구체화하세요.

먼저 기본값으로 한 번 실행하세요. 그다음 표시된 값 하나만 바꾸고, 달라진 결과를 아래에 적습니다.


In [ ]:
team_limit = "메뉴 이름에 적히지 않은 재료와 실제 학생 만족도를 알 수 없다."
print("우리 팀이 발표할 한계:", team_limit)


### 내가 본 변화

- 내가 바꾼 값:  
- 화면에서 달라진 것:  
- 내 설명:


## 확인 문제

1. 좋은 테스트 예상은 왜 구체적이어야 하나요?
2. 모델 카드에 잘하는 것뿐 아니라 한계도 쓰는 이유는 무엇인가요?
3. 이 추천기를 학생의 건강 판단에 쓰면 안 되는 이유 두 가지를 말해 보세요.


## 정답과 해설


1. 실행 결과가 통과인지 실패인지 분명히 판정하기 위해서입니다.<br>
2. 사용자가 결과의 범위와 위험을 알고 과장해서 사용하지 않도록 하기 위해서입니다.<br>
3. 데이터가 5행으로 적고, 개인 건강 정보·숨은 재료·실제 만족도를 사용하지 않았기 때문입니다.


## 핵심 정리

- 테스트는 예상·실행·판정을 구체적으로 기록한다.
- 입력 경계와 안전 제외 규칙도 시험한다.
- 모델 카드는 AI의 목적과 한계, 금지 사용을 함께 공개한다.

### 다음 장

08장에서는 완성된 프로젝트를 문제부터 한계까지 8개 구간으로 발표합니다.


In [ ]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))
